Celula 1 – Încărcarea datasetului pentru antrenare

In [28]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("../Data/processed")

# Incarcam seturile pregatite
train_df = pd.read_parquet(DATA_DIR / "train.parquet")
val_df = pd.read_parquet(DATA_DIR / "validation.parquet")

# Incarcam maparea categoriilor
with open(DATA_DIR / "label2id.json", encoding="utf-8") as f:
    label2id = json.load(f)

with open(DATA_DIR / "id2label.json", encoding="utf-8") as f:
    id2label = {
        int(k): v
        for k, v in json.load(f).items()
    }

NUM_LABELS = len(label2id)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Numar categorii:", NUM_LABELS)

print("\nPrimul document:")
print("CELEX ID:", train_df.iloc[0]["celex_id"])
print("Etichete:", train_df.iloc[0]["labels"])
print("Vector multi-hot:", train_df.iloc[0]["labels_encoded"])


Training: 15920
Validation: 5000
Numar categorii: 21

Primul document:
CELEX ID: 32006D0213
Etichete: ['100160' '100155' '100158' '100147' '100149']
Vector multi-hot: [0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0.]


Celula 2 – Încărcarea tokenizerului XLM-RoBERTa

Acum pregătim tokenizerul pentru primul nostru encoder.

In [29]:

from transformers import AutoTokenizer

MODEL_NAME = "FacebookAI/xlm-roberta-base"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Model:", MODEL_NAME)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Lungime maxima:", tokenizer.model_max_length)
print("Tokenuri speciale:", tokenizer.num_special_tokens_to_add(pair=False))

assert tokenizer.model_max_length == MAX_LENGTH


Model: FacebookAI/xlm-roberta-base
Tokenizer: XLMRobertaTokenizer
Lungime maxima: 512
Tokenuri speciale: 2


Celula 3 – Construirea clasei Dataset

De ce avem nevoie de această clasă?

În prezent, documentele sunt într-un tabel Pandas (train_df), dar modelul PyTorch trebuie să primească date într-un format numeric.

Vom crea o clasă care, pentru fiecare document:
1. Preia textul original.
2. Îl transformă în tokenuri folosind XLM-RoBERTa.
3. Îl trunchiază la maximum 512 tokenuri.
4. Returnează identificatorii tokenurilor, masca de atenție și cele 21 de etichete.

In [30]:

import torch
from torch.utils.data import Dataset

class LegalDocumentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["text"].tolist()
        self.labels = np.stack(
            dataframe["labels_encoded"].to_numpy()
        ).astype(np.float32)

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[index],
                dtype=torch.float32
            )
        }


train_dataset = LegalDocumentDataset(
    train_df, tokenizer, MAX_LENGTH
)

val_dataset = LegalDocumentDataset(
    val_df, tokenizer, MAX_LENGTH
)

print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))

sample = train_dataset[0]

print("\nPrimul document:")
print("Forma input_ids:", sample["input_ids"].shape)
print("Forma attention_mask:", sample["attention_mask"].shape)
print("Forma labels:", sample["labels"].shape)
print("Numar etichete active:", int(sample["labels"].sum()))


Training: 15920
Validation: 5000

Primul document:
Forma input_ids: torch.Size([512])
Forma attention_mask: torch.Size([512])
Forma labels: torch.Size([21])
Numar etichete active: 5


Celula 4 – Construirea batch-urilor cu dynamic padding

De ce avem nevoie de acest pas?
La antrenare, modelul nu procesează neapărat un singur document, ci mai multe simultan. Acest grup se numește batch.

Problema este că documentele au lungimi diferite.

De exemplu:
- Document A: 512 tokenuri
- Document B: 350 tokenuri
- Document C: 200 tokenuri
Pentru a le grupa într-un tensor, toate secvențele trebuie să aibă aceeași lungime.

Vom adăuga tokenuri de completare (padding) doar până la lungimea celui mai lung document din batch, nu obligatoriu până la 512.

In [31]:

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

# Verificam folosind trei documente
samples = [
    train_dataset[i]
    for i in range(3)
]

batch = data_collator(samples)

print("Forma input_ids:", batch["input_ids"].shape)
print("Forma attention_mask:", batch["attention_mask"].shape)
print("Forma labels:", batch["labels"].shape)

print("\nLungimi originale:")
for sample in samples:
    print(len(sample["input_ids"]))

print("\nLungimi reale dupa padding:")
print(batch["attention_mask"].sum(dim=1).tolist())

print("\nForma batch-ului:", batch["input_ids"].shape)


Forma input_ids: torch.Size([3, 512])
Forma attention_mask: torch.Size([3, 512])
Forma labels: torch.Size([3, 21])

Lungimi originale:
512
512
512

Lungimi reale dupa padding:
[512, 512, 512]

Forma batch-ului: torch.Size([3, 512])


Celula 5 – Verificăm dynamic padding pe documente de lungimi diferite

Vom selecta automat un document mai scurt de 512 tokenuri și unul cu exact 512 tokenuri.

In [32]:

# Cautam un document cu mai putin de 512 tokenuri
short_index = next(
    i for i in range(len(train_dataset))
    if len(train_dataset[i]["input_ids"]) < 512
)

# Cautam un document de 512 tokenuri
long_index = next(
    i for i in range(len(train_dataset))
    if len(train_dataset[i]["input_ids"]) == 512
)

short_sample = train_dataset[short_index]
long_sample = train_dataset[long_index]

print("Lungime document scurt:",
      len(short_sample["input_ids"]))

print("Lungime document lung:",
      len(long_sample["input_ids"]))

# Construim un batch cu cele doua documente
mixed_batch = data_collator([
    short_sample,
    long_sample
])

print("\nForma batch:", mixed_batch["input_ids"].shape)

print(
    "Tokenuri reale per document:",
    mixed_batch["attention_mask"].sum(dim=1).tolist()
)

print(
    "Tokenuri padding per document:",
    (mixed_batch["attention_mask"] == 0).sum(dim=1).tolist()
)

print("Forma etichetelor:", mixed_batch["labels"].shape)


Lungime document scurt: 386
Lungime document lung: 512

Forma batch: torch.Size([2, 512])
Tokenuri reale per document: [386, 512]
Tokenuri padding per document: [126, 0]
Forma etichetelor: torch.Size([2, 21])


Celula 6 – Încărcarea encoderului XLM-RoBERTa pentru clasificare multi-label

Până acum am încărcat doar tokenizerul. Acum încărcăm modelul propriu-zis, preantrenat pe texte multilingve, și îi atașăm un strat de clasificare pentru cele 21 de categorii.

In [33]:

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print("Arhitectura:", model.config.model_type)
print("Numar categorii:", model.config.num_labels)
print("Tip clasificare:", model.config.problem_type)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"\nParametri totali: {total_params:,}")
print(f"Parametri antrenabili: {trainable_params:,}")

assert model.config.num_labels == 21
assert model.config.problem_type == "multi_label_classification"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Arhitectura: xlm-roberta
Numar categorii: 21
Tip clasificare: multi_label_classification

Parametri totali: 278,059,797
Parametri antrenabili: 278,059,797


Pe GPU-ul meu cu 7,96 GB VRAM, va trebui să fim atenți la consumul de memorie. De aceea vom testa separat dacă modelul poate efectua un pas de antrenare, înainte de a lansa antrenarea completă.

Celula 7 – Verificăm un forward pass

Ce este un forward pass?
Este trecerea datelor prin model pentru a calcula predicții.

În acest moment nu actualizăm parametrii. Vrem doar să verificăm dacă modelul primește corect documentele și etichetele pregătite.

Pentru a evita ocuparea inutilă a memoriei GPU, efectuăm acest test pe CPU, cu un singur document.

In [34]:

import torch

# Selectam un singur document
sample = train_dataset[0]

# Adaugam dimensiunea batch-ului
inputs = {
    "input_ids": sample["input_ids"].unsqueeze(0),
    "attention_mask": sample["attention_mask"].unsqueeze(0),
    "labels": sample["labels"].unsqueeze(0)
}

# Executam modelul fara calcularea gradientilor
model.eval()

with torch.no_grad():
    outputs = model(**inputs)

print("Forma logits:", outputs.logits.shape)
print("Loss:", outputs.loss.item())

# Transformam logits in probabilitati
probabilities = torch.sigmoid(outputs.logits)

print("Forma probabilitati:", probabilities.shape)
print("Primele 5 probabilitati:", probabilities[0, :5].tolist())

assert outputs.logits.shape == (1, 21)
assert torch.isfinite(outputs.loss)


Forma logits: torch.Size([1, 21])
Loss: 0.6965706944465637
Forma probabilitati: torch.Size([1, 21])
Primele 5 probabilitati: [0.5307585597038269, 0.44815918803215027, 0.5123278498649597, 0.43580323457717896, 0.5609490871429443]


Primele cinci probabilități sunt aproximativ 0,46, 0,57, 0,56, 0,55 și 0,53.

Nu înseamnă că modelul clasifică bine documentele! Clasificatorul este încă neantrenat, iar aceste probabilități nu reprezintă o performanță relevantă.

Valoarea loss de 0,7090 ne arată doar că putem calcula eroarea. Nu putem spune dacă este bună sau rea înainte de antrenare și evaluare.

Următorul pas: putem antrena modelul pe GPU?
Acum trebuie să verificăm dacă GPU-ul tău NVIDIA RTX 5070 Laptop, cu 7,96 GB VRAM, poate executa un pas de antrenare.

Vom testa un singur pas, nu o epocă întreagă.

Un pas de antrenare presupune:
1. Forward pass: modelul calculează predicțiile și loss-ul.
2. Backward pass: PyTorch calculează gradienții.
3. Optimizer step: algoritmul de optimizare actualizează parametrii modelului.
Pentru testul nostru vom executa doar primii doi pași. Astfel, verificăm memoria necesară fără să modificăm greutățile modelului.

Celula 8 – Test de memorie GPU

Vom folosi un singur document de 512 tokenuri și mixed precision (float16) pentru a reduce consumul de memorie.

In [35]:

import torch
import gc

assert torch.cuda.is_available(), "CUDA nu este disponibil!"

device = torch.device("cuda")

# Eliberam memoria GPU neutilizata
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Mutam modelul pe GPU
model = model.to(device)
model.train()

# Un singur document
sample = train_dataset[0]

inputs = {
    "input_ids": sample["input_ids"].unsqueeze(0).to(device),
    "attention_mask": sample["attention_mask"].unsqueeze(0).to(device),
    "labels": sample["labels"].unsqueeze(0).to(device)
}

# Curatam eventualii gradienti existenti
model.zero_grad(set_to_none=True)

# Forward pass cu mixed precision
with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(**inputs)
    loss = outputs.loss

print("Loss:", loss.item())

# Backward pass
loss.backward()

print("Backward pass: reusit")

print(
    "Memorie GPU alocata:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "Memorie GPU maxima:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

# Eliberam gradientii dupa test
model.zero_grad(set_to_none=True)


Loss: 0.6981569528579712
Backward pass: reusit
Memorie GPU alocata: 3.11 GB
Memorie GPU maxima: 3.11 GB


Celula 9 – Testăm un pas complet de antrenare

In [36]:

import gc
import torch

device = torch.device("cuda")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()
model.zero_grad(set_to_none=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5
)

scaler = torch.amp.GradScaler("cuda")

sample = train_dataset[0]

inputs = {
    "input_ids": sample["input_ids"].unsqueeze(0).to(device),
    "attention_mask": sample["attention_mask"].unsqueeze(0).to(device),
    "labels": sample["labels"].unsqueeze(0).to(device)
}

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(**inputs)
    loss = outputs.loss

print("Loss inainte de actualizare:", loss.item())

scaler.scale(loss).backward()
scaler.unscale_(optimizer)

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

old_scale = scaler.get_scale()

scaler.step(optimizer)
scaler.update()

new_scale = scaler.get_scale()

print("Scara FP16 inainte:", old_scale)
print("Scara FP16 dupa:", new_scale)
print("Actualizare efectuata:", new_scale >= old_scale)

print(
    "Memorie GPU maxima:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

optimizer.zero_grad(set_to_none=True)


Loss inainte de actualizare: 0.7066634893417358
Scara FP16 inainte: 65536.0
Scara FP16 dupa: 65536.0
Actualizare efectuata: True
Memorie GPU maxima: 6.22 GB


Celula 10 – Reinițializarea controlată a modelului

Vom fixa un seed și vom reîncărca modelul preantrenat. Astfel, eliminăm efectul pasului experimental de optimizare și inițializăm clasificatorul într-un mod reproductibil.

In [41]:

import gc
import random
import numpy as np
import torch

from transformers import AutoModelForSequenceClassification

SEED = 42

# Eliberam modelul si optimizatorul folosite la test
if "optimizer" in globals():
    del optimizer

if "scaler" in globals():
    del scaler

if "model" in globals():
    del model

if "outputs" in globals():
    del outputs

if "loss" in globals():
    del loss

gc.collect()
torch.cuda.empty_cache()

# Fixam seed-ul
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reincarcam checkpoint-ul original
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)

print("Model reincarcat:", MODEL_NAME)
print("Seed:", SEED)
print("Numar categorii:", model.config.num_labels)
print("Dispozitiv initial:", next(model.parameters()).device)

print(
    "Memorie GPU alocata dupa reincarcare:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model reincarcat: FacebookAI/xlm-roberta-base
Seed: 42
Numar categorii: 21
Dispozitiv initial: cpu
Memorie GPU alocata dupa reincarcare: 1.04 GB


Celula 11 – Definirea funcției de evaluare

In [38]:

import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Convertim logits in probabilitati
    probabilities = 1 / (1 + np.exp(-logits))

    # Prag initial de clasificare
    predictions = (probabilities >= 0.5).astype(int)

    # Etichetele reale
    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(
            labels, predictions,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            labels, predictions,
            average="macro",
            zero_division=0
        ),
        "micro_precision": precision_score(
            labels, predictions,
            average="micro",
            zero_division=0
        ),
        "micro_recall": recall_score(
            labels, predictions,
            average="micro",
            zero_division=0
        )
    }

print("Functia compute_metrics a fost definita.")


Functia compute_metrics a fost definita.


Celula 12 – Verificarea metricilor pe un exemplu controlat

In [39]:

import numpy as np

# Doua documente, fiecare cu 21 de categorii
true_labels = np.zeros((2, 21), dtype=int)
predicted_labels = np.zeros((2, 21), dtype=int)

# Documentul 1
true_labels[0, [3, 5]] = 1
predicted_labels[0, [3, 5]] = 1

# Documentul 2
true_labels[1, [7, 13]] = 1
predicted_labels[1, [7, 18]] = 1

# Convertim predictiile in logits,
# folosind probabilitati de 0.9 si 0.1
probabilities = np.where(
    predicted_labels == 1, 0.9, 0.1
)

logits = np.log(
    probabilities / (1 - probabilities)
)

results = compute_metrics((logits, true_labels))

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

assert np.isclose(results["micro_precision"], 0.75)
assert np.isclose(results["micro_recall"], 0.75)
assert np.isclose(results["micro_f1"], 0.75)

print("\nVerificarea metricilor: reusita")


micro_f1: 0.7500
macro_f1: 0.1429
micro_precision: 0.7500
micro_recall: 0.7500

Verificarea metricilor: reusita


Celula 13 – Antrenarea XLM-RoBERTa

In [42]:

from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "../Models/xlm_roberta_finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Antrenare completa
    num_train_epochs=3,

    # Batch efectiv = 1 x 8 = 8
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Optimizare
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=0.1,
    max_grad_norm=1.0,

    # Reducerea consumului de memorie
    fp16=True,
    gradient_checkpointing=True,

    # Loss si metrici dupa fiecare epoca
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",

    # Selectam checkpoint-ul cu cel mai bun Micro-F1
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    # Pastram maximum doua checkpoint-uri
    save_total_limit=2,

    # Reproductibilitate
    seed=42,
    data_seed=42,

    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Configuratia antrenarii este valida.")
print("Epoci:", training_args.num_train_epochs)
print("Batch efectiv:", training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)
print("Warmup:", training_args.warmup_steps)
print("Evaluare:", training_args.eval_strategy)
print("Logging:", training_args.logging_strategy)


/home/emilovici/projects/NLP-Master/.venv311/lib/python3.11/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Configuratia antrenarii este valida.
Epoci: 3
Batch efectiv: 8
Warmup: 0.1
Evaluare: IntervalStrategy.EPOCH
Logging: IntervalStrategy.EPOCH


Celula 14 – Antrenarea modelului

In [43]:

# Pornim antrenarea completa
train_result = trainer.train()

print("\n=== ANTRENARE FINALIZATA ===")
print(train_result.metrics)

print("\n=== REZULTATE PE EPOCI ===")

import pandas as pd

history = trainer.state.log_history

train_logs = [
    entry for entry in history
    if "loss" in entry and "eval_loss" not in entry
]

eval_logs = [
    entry for entry in history
    if "eval_loss" in entry
]

epoch_results = pd.DataFrame([
    {
        "Epoch": eval_entry["epoch"],
        "Train Loss": train_entry["loss"],
        "Validation Loss": eval_entry["eval_loss"],
        "Micro-F1": eval_entry["eval_micro_f1"],
        "Macro-F1": eval_entry["eval_macro_f1"]
    }
    for train_entry, eval_entry in zip(train_logs, eval_logs)
])

print(epoch_results.to_string(index=False))

print("\nCel mai bun checkpoint:")
print(trainer.state.best_model_checkpoint)

print("\nCel mai bun Micro-F1:")
print(trainer.state.best_metric)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,2.426382,0.205762,0.766181,0.515451,0.867823,0.685852
2,1.499963,0.183929,0.792875,0.577955,0.870511,0.727953
3,1.284821,0.174636,0.812881,0.644635,0.868949,0.763611


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== ANTRENARE FINALIZATA ===
{'train_runtime': 5247.9387, 'train_samples_per_second': 9.101, 'train_steps_per_second': 1.138, 'total_flos': 1.1995228040651694e+16, 'train_loss': 1.7370554759474455, 'epoch': 3.0}

=== REZULTATE PE EPOCI ===
 Epoch  Train Loss  Validation Loss  Micro-F1  Macro-F1
   1.0    2.426382         0.205762  0.766181  0.515451
   2.0    1.499963         0.183929  0.792875  0.577955
   3.0    1.284821         0.174636  0.812881  0.644635

Cel mai bun checkpoint:
../Models/xlm_roberta_finetuned/checkpoint-5970

Cel mai bun Micro-F1:
0.8128814741614835


Am antrenat modelul cu 3 epoci(adica se va parcurge setul de training de trei ori). După fiecare epocă, se vor calcula metricile pe validation și se va salva un checkpoint.

La final, load_best_model_at_end=True va reîncărca automat modelul cu cel mai bun Micro-F1.

Atenție: această celulă nu salvează încă o copie separată a modelului final în best_model. 
După ce analizăm rezultatele, vom salva explicit modelul selectat și tokenizerul.

Antrenarea XLM-RoBERTa s-a încheiat cu succes, iar rezultatele arată o îmbunătățire constantă a performanței pe validation în toate cele trei epoci. Este primul nostru rezultat experimental real pentru proiectul NLP!

Totuși, am observat și un aspect care trebuie verificat: diferența neobișnuit de mare dintre Train Loss și Validation Loss. 

Train Loss este mult mai mare decât Validation Loss, chiar și după trei epoci. O asemenea diferență nu trebuie interpretată automat drept underfitting sau overfitting.

Înainte de a decide dacă prelungim antrenarea, propun să evaluăm checkpoint-ul selectat pe un subset de 256 de documente din training, în același mod în care evaluăm validation.

Astfel obținem un loss calculat în modul eval, fără actualizarea parametrilor.

Celula 15 – Verificarea Training Loss

In [44]:

from torch.utils.data import Subset

# Alegem un subset reproductibil din training
rng = np.random.default_rng(42)

indices = rng.choice(
    len(train_dataset),
    size=256,
    replace=False
)

train_subset = Subset(
    train_dataset,
    indices.tolist()
)

# Evaluam checkpoint-ul selectat, fara antrenare
train_subset_results = trainer.evaluate(
    eval_dataset=train_subset,
    metric_key_prefix="train_subset"
)

print("\n=== EVALUARE TRAINING SUBSET ===")

for metric, value in train_subset_results.items():
    print(f"{metric}: {value}")

print("\nValidation Loss epoca 3:", 0.174636)


Training Loss,Validation Loss,Epoch,Subset Loss,Subset Micro F1,Subset Macro F1,Subset Micro Precision,Subset Micro Recall
1.284821,No log,3,0.128575,0.859837,0.704930,0.916890,0.809467



=== EVALUARE TRAINING SUBSET ===
train_subset_loss: 0.12857535481452942
train_subset_micro_f1: 0.8598365807668134
train_subset_macro_f1: 0.7049296973988518
train_subset_micro_precision: 0.9168900804289544
train_subset_micro_recall: 0.8094674556213017

Validation Loss epoca 3: 0.174636


Acum avem o informație importantă: loss-ul calculat pe un subset de training este mai mic decât Validation Loss, ceea ce este plauzibil pentru un model care a învățat din datele de antrenare.

Diferența de loss este acum mult mai rezonabilă.
- Training subset loss: 0,1286
- Validation loss: 0,1746
Ambele au fost calculate prin trainer.evaluate(), deci folosind aceeași metodă de evaluare.

Acest rezultat susține ipoteza că valorile Train Loss afișate în timpul antrenării (2,4264, 1,5000, 1,2848) nu sunt direct comparabile numeric cu Validation Loss.

Nu am demonstrat încă mecanismul exact care a produs diferența, așa că nu vom prezenta acele valori drept o curbă de loss comparabilă cu validation.

Celula 16 – Verificarea checkpoint-ului (verificăm efectiv ce fișiere au fost salvate, ca sa vedem daca putem continua antrenarea de la chekpointul salvat sau trebuie sa configuram o noua antrenare de la 0)

In [45]:

from pathlib import Path

checkpoint_path = Path(
    "../Models/xlm_roberta_finetuned/checkpoint-5970"
)

print("Checkpoint exista:", checkpoint_path.exists())

if checkpoint_path.exists():
    print("\nFisiere din checkpoint:")

    for file in sorted(checkpoint_path.iterdir()):
        print(file.name)

    required = [
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt"
    ]

    print("\nFisiere necesare pentru reluare:")

    for name in required:
        print(
            f"{name}:",
            (checkpoint_path / name).exists()
        )


Checkpoint exista: True

Fisiere din checkpoint:
config.json
model.safetensors
optimizer.pt
rng_state.pth
scaler.pt
scheduler.pt
tokenizer.json
tokenizer_config.json
trainer_state.json
training_args.bin

Fisiere necesare pentru reluare:
trainer_state.json: True
optimizer.pt: True
scheduler.pt: True


Celula 17 – Verificarea progresului și a learning rate-ului

In [46]:

import json
from pathlib import Path

checkpoint_path = Path(
    "../Models/xlm_roberta_finetuned/checkpoint-5970"
)

with open(
    checkpoint_path / "trainer_state.json",
    encoding="utf-8"
) as f:
    state = json.load(f)

print("Epoca finala:", state["epoch"])
print("Pasi efectuati:", state["global_step"])
print("Total pasi planificati:", state["max_steps"])

print("\nUltimele valori Learning Rate:")

lr_logs = [
    entry
    for entry in state["log_history"]
    if "learning_rate" in entry
]

for entry in lr_logs[-5:]:
    print(
        "Epoch:", entry.get("epoch"),
        "| Step:", entry.get("step"),
        "| LR:", entry["learning_rate"]
    )

print("\nCel mai bun checkpoint:",
      state["best_model_checkpoint"])

print("Cel mai bun Micro-F1:",
      state["best_metric"])


Epoca finala: 3.0
Pasi efectuati: 5970
Total pasi planificati: 5970

Ultimele valori Learning Rate:
Epoch: 1.0 | Step: 1990 | LR: 1.4818537130094921e-05
Epoch: 2.0 | Step: 3980 | LR: 7.411129722687512e-06
Epoch: 3.0 | Step: 5970 | LR: 3.7223152801042253e-09

Cel mai bun checkpoint: ../Models/xlm_roberta_finetuned/checkpoint-5970
Cel mai bun Micro-F1: 0.8128814741614835


Acum știm exact de ce nu ar fi corect să continuăm antrenarea pur și simplu din checkpoint, cu aceeași configurație.

Learning rate-ul a scăzut progresiv, ajungând la final la aproximativ 3,72 × 10⁻⁹, practic zero.

Acest lucru este normal: schedulerul a fost configurat pentru 5.970 de pași și a ajuns la finalul programului său.

Dacă am relua antrenarea păstrând neschimbat schedulerul, modelul ar face actualizări extrem de mici sau inexistente.

O sa urmez următoarea abordare:
- Păstrăm separat checkpoint-ul obținut după primele 3 epoci.
- Pornim o fază suplimentară de fine-tuning de la aceste greutăți, cu un optimizator și un scheduler noi.
- Folosim un learning rate inițial mai mic, de exemplu 5e-6, deoarece modelul a fost deja adaptat la sarcina noastră.
- Configurăm maximum 3 epoci suplimentare, cu Early Stopping (patience=2).
- Comparăm cel mai bun rezultat al continuării cu Micro-F1 = 0,812881 obținut până acum.

Celula 18 – Salvăm modelul obținut după 3 epoci

In [47]:

from pathlib import Path

BASELINE_DIR = Path("../Models/xlm_roberta_3epochs")

# Salvam modelul selectat dupa primele 3 epoci
trainer.save_model(str(BASELINE_DIR))

# Salvam explicit tokenizerul
tokenizer.save_pretrained(str(BASELINE_DIR))

print("Model salvat in:", BASELINE_DIR.resolve())

print("\nFisiere salvate:")
for file in sorted(BASELINE_DIR.iterdir()):
    print(file.name)

print("\nMicro-F1 validation:", trainer.state.best_metric)
print("Checkpoint original:", trainer.state.best_model_checkpoint)

assert (BASELINE_DIR / "model.safetensors").exists()
assert (BASELINE_DIR / "config.json").exists()

print("\nSalvare verificata cu succes!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model salvat in: /home/emilovici/projects/NLP-Master/Project_multi_eurlex/Models/xlm_roberta_3epochs

Fisiere salvate:
config.json
model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin

Micro-F1 validation: 0.8128814741614835
Checkpoint original: ../Models/xlm_roberta_finetuned/checkpoint-5970

Salvare verificata cu succes!


Celula 19 – Încărcarea modelului pentru continuarea antrenării

In [48]:

import gc
import torch

from transformers import (
    AutoModelForSequenceClassification,
    set_seed
)

# Eliberam obiectele vechi de antrenare
if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

# Seed pentru reproductibilitate
set_seed(42)

CONTINUE_MODEL_DIR = "../Models/xlm_roberta_3epochs"

# Incarcam greutatile deja antrenate
model = AutoModelForSequenceClassification.from_pretrained(
    CONTINUE_MODEL_DIR
)

print("Model incarcat din:", CONTINUE_MODEL_DIR)
print("Arhitectura:", model.config.model_type)
print("Numar categorii:", model.config.num_labels)
print("Tip clasificare:", model.config.problem_type)
print("Dispozitiv initial:", next(model.parameters()).device)

assert model.config.num_labels == 21
assert model.config.problem_type == "multi_label_classification"

print("\nModel pregatit pentru faza suplimentara.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model incarcat din: ../Models/xlm_roberta_3epochs
Arhitectura: xlm-roberta
Numar categorii: 21
Tip clasificare: multi_label_classification
Dispozitiv initial: cpu

Model pregatit pentru faza suplimentara.


Celula 20 – Configurarea antrenării cu Early Stopping

Voi folosi maximum 3 epoci suplimentare, un learning rate mai mic (5e-6) și oprire anticipată dacă Micro-F1 nu se îmbunătățește timp de două evaluări consecutive.

In [49]:

from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

CONTINUE_OUTPUT_DIR = "../Models/xlm_roberta_continued"

continue_args = TrainingArguments(
    output_dir=CONTINUE_OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_steps=0.1,
    max_grad_norm=1.0,

    fp16=True,
    gradient_checkpointing=True,

    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,

    seed=42,
    data_seed=42,

    dataloader_num_workers=0,
    report_to="none"
)

continue_trainer = Trainer(
    model=model,
    args=continue_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0
        )
    ]
)

print("Configuratia continuarii este valida.")
print("Epoci suplimentare maxime:", continue_args.num_train_epochs)
print("Learning rate:", continue_args.learning_rate)
print(
    "Batch efectiv:",
    continue_args.per_device_train_batch_size *
    continue_args.gradient_accumulation_steps
)
print("Early Stopping patience: 2")
print("Criteriu selectie:", continue_args.metric_for_best_model)


/home/emilovici/projects/NLP-Master/.venv311/lib/python3.11/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Configuratia continuarii este valida.
Epoci suplimentare maxime: 3
Learning rate: 5e-06
Batch efectiv: 8
Early Stopping patience: 2
Criteriu selectie: micro_f1


Ce este important:
1. Nu pornim de la zero. Modelul conține deja greutățile învățate în primele trei epoci. Inițializăm doar un nou proces de optimizare.

2. Early Stopping verifică Micro-F1 după fiecare epocă. Dacă nu apare o îmbunătățire timp de două evaluări consecutive, antrenarea se oprește anticipat.

3. Avem o limitare a acestei configurații: Early Stopping compară rezultatele dintre epocile fazei suplimentare. Nu compară automat prima evaluare cu Micro-F1 = 0.812881 din experimentul anterior.

De aceea, la final vom compara explicit rezultatele celor două faze și vom păstra modelul care a obținut cel mai bun Micro-F1 pe validation.

4. Trei epoci suplimentare sunt limita maximă, nu o garanție că Early Stopping se va activa. Cu patience=2 și evaluare doar după fiecare epocă, este posibil ca antrenarea să ajungă la toate cele trei epoci.

Celula 21 – Continuarea antrenării cu Early Stopping

In [50]:

# Continuam antrenarea modelului deja fine-tuned
continue_result = continue_trainer.train()

print("\n=== ANTRENARE SUPLIMENTARA FINALIZATA ===")
print(continue_result.metrics)

# Extragem rezultatele pe epoci
history = continue_trainer.state.log_history

train_logs = [
    entry for entry in history
    if "loss" in entry and "eval_loss" not in entry
]

eval_logs = [
    entry for entry in history
    if "eval_loss" in entry
]

epoch_results = pd.DataFrame([
    {
        "Epoca suplimentara": eval_entry["epoch"],
        "Train Loss raportat": train_entry["loss"],
        "Validation Loss": eval_entry["eval_loss"],
        "Micro-F1": eval_entry["eval_micro_f1"],
        "Macro-F1": eval_entry["eval_macro_f1"]
    }
    for train_entry, eval_entry in zip(train_logs, eval_logs)
])

print("\n=== REZULTATE PE EPOCI ===")
print(epoch_results.to_string(index=False))

print("\nCel mai bun checkpoint al fazei suplimentare:")
print(continue_trainer.state.best_model_checkpoint)

print("\nCel mai bun Micro-F1 al fazei suplimentare:")
print(continue_trainer.state.best_metric)

print("\nMicro-F1 model initial (3 epoci):")
print(0.8128814741614835)


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,1.196773,0.172788,0.812875,0.646656,0.871137,0.761918
2,1.080910,0.174381,0.813714,0.653532,0.862896,0.769836
3,1.051548,0.172744,0.817246,0.663576,0.860829,0.777863


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== ANTRENARE SUPLIMENTARA FINALIZATA ===
{'train_runtime': 5284.151, 'train_samples_per_second': 9.038, 'train_steps_per_second': 1.13, 'total_flos': 1.1995228040651694e+16, 'train_loss': 1.1097437464012772, 'epoch': 3.0}

=== REZULTATE PE EPOCI ===
 Epoca suplimentara  Train Loss raportat  Validation Loss  Micro-F1  Macro-F1
                1.0             1.196773         0.172788  0.812875  0.646656
                2.0             1.080910         0.174381  0.813714  0.653532
                3.0             1.051548         0.172744  0.817246  0.663576

Cel mai bun checkpoint al fazei suplimentare:
../Models/xlm_roberta_continued/checkpoint-5970

Cel mai bun Micro-F1 al fazei suplimentare:
0.8172456326554028

Micro-F1 model initial (3 epoci):
0.8128814741614835


Ce observăm?
1. Continuarea antrenării a fost utilă. Micro-F1 a crescut cu 0,00436, iar Macro-F1 cu 0,01894.

2. Îmbunătățirea este mai pronunțată pentru Macro-F1. Acest rezultat este încurajator pentru performanța medie pe categorii, dar nu ne spune încă exact care categorii s-au îmbunătățit.

3. Early Stopping nu s-a activat. Antrenarea a ajuns la toate cele trei epoci suplimentare, iar cel mai bun Micro-F1 s-a obținut la ultima epocă.

4. Validation Loss nu a scăzut monoton. A crescut puțin în epoca 5, apoi a scăzut în epoca 6. Este un exemplu bun despre motivul pentru care nu oprim automat antrenarea la prima creștere a loss-ului.

Nu avem dovezi clare de deteriorare a performanței pe validation, dar îmbunătățirea Micro-F1 din a doua fază este relativ mică.

Celula 22 - Selectăm checkpoint-ul de la epoca 6 drept candidatul final XLM-RoBERTa, deoarece a obținut cel mai bun Micro-F1 pe validation, si il salvam explicit intr-un folder.

In [51]:

from pathlib import Path
import json

FINAL_XLM_DIR = Path("../Models/xlm_roberta_best")

# Salvam modelul selectat automat dupa validation Micro-F1
continue_trainer.save_model(str(FINAL_XLM_DIR))
tokenizer.save_pretrained(str(FINAL_XLM_DIR))

# Salvam informatiile experimentului
experiment_info = {
    "model": "FacebookAI/xlm-roberta-base",
    "task": "multi_label_classification",
    "num_labels": 21,
    "max_length": 512,
    "training_documents": 15920,
    "validation_documents": 5000,
    "initial_epochs": 3,
    "additional_epochs": 3,
    "additional_learning_rate": 5e-6,
    "best_validation_micro_f1": continue_trainer.state.best_metric,
    "best_checkpoint": continue_trainer.state.best_model_checkpoint
}

with open(
    FINAL_XLM_DIR / "experiment_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        experiment_info,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Model salvat:", FINAL_XLM_DIR.resolve())

print("\nFisiere:")
for file in sorted(FINAL_XLM_DIR.iterdir()):
    print(file.name)

print("\nMicro-F1 validation:",
      continue_trainer.state.best_metric)

assert (FINAL_XLM_DIR / "model.safetensors").exists()
assert (FINAL_XLM_DIR / "experiment_info.json").exists()

print("\nSalvare verificata!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model salvat: /home/emilovici/projects/NLP-Master/Project_multi_eurlex/Models/xlm_roberta_best

Fisiere:
config.json
experiment_info.json
model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin

Micro-F1 validation: 0.8172456326554028

Salvare verificata!


Este un moment bun să facem curățenie în Models, înainte să trecem la Legal-XLM-Longformer.

Din experimentele noastre, avem mai multe versiuni XLM-RoBERTa, dar doar una este necesară pentru evaluarea finală și demo: xlm_roberta_best, cu Micro-F1 = 0,817246 pe validation.

In [52]:

from pathlib import Path

MODELS_DIR = Path("../Models")

print("Continutul folderului Models:\n")

for folder in sorted(MODELS_DIR.iterdir()):
    if folder.is_dir():
        files = list(folder.rglob("*"))
        total_size = sum(
            f.stat().st_size
            for f in files
            if f.is_file()
        )

        print(
            f"{folder.name}/ "
            f"- {total_size / (1024**3):.2f} GB"
        )
    else:
        print(folder.name)


Continutul folderului Models:

xlm_roberta_3epochs/ - 1.05 GB
xlm_roberta_best/ - 1.05 GB
xlm_roberta_continued/ - 6.25 GB
xlm_roberta_finetuned/ - 6.25 GB


Arhivarea rezultatelor antrenării
Vom extrage istoricul direct din fișierele trainer_state.json ale checkpoint-urilor, nu din valori copiate manual.

Vom salva rezultatele în results/xlm_roberta_training_history.csv, ca dupa sa pot sterge modelele din Models pe care nu o sa le folosesc la comparatie, il las doar pe cel mia bun checkpoint.

In [54]:

import json
import pandas as pd
from pathlib import Path

MODELS_DIR = Path("../Models")
RESULTS_DIR = Path("../Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

phases = [
    ("initial", "xlm_roberta_finetuned", 0),
    ("continued", "xlm_roberta_continued", 3)
]

rows = []

for phase, folder, offset in phases:
    state_path = (
        MODELS_DIR / folder / "checkpoint-5970" /
        "trainer_state.json"
    )

    with open(state_path, encoding="utf-8") as f:
        state = json.load(f)

    train_logs = [
        entry for entry in state["log_history"]
        if "loss" in entry and "eval_loss" not in entry
    ]

    eval_logs = [
        entry for entry in state["log_history"]
        if "eval_loss" in entry
    ]

    assert len(train_logs) == len(eval_logs) == 3

    for train_entry, eval_entry in zip(train_logs, eval_logs):
        assert train_entry["epoch"] == eval_entry["epoch"]

        rows.append({
            "phase": phase,
            "epoch": int(eval_entry["epoch"]) + offset,
            "train_loss_reported": train_entry["loss"],
            "validation_loss": eval_entry["eval_loss"],
            "micro_f1": eval_entry["eval_micro_f1"],
            "macro_f1": eval_entry["eval_macro_f1"],
            "learning_rate": train_entry.get("learning_rate")
        })

history_df = pd.DataFrame(rows)

output_path = RESULTS_DIR / "xlm_roberta_training_history.csv"

history_df.to_csv(output_path, index=False)

# Verificam ca fisierul salvat poate fi citit
saved_history = pd.read_csv(output_path)

assert len(saved_history) == 6
assert saved_history["epoch"].tolist() == [1, 2, 3, 4, 5, 6]

print(saved_history.to_string(index=False))
print("\nFisier salvat:", output_path.resolve())
print("Numar epoci arhivate:", len(saved_history))


    phase  epoch  train_loss_reported  validation_loss  micro_f1  macro_f1  learning_rate
  initial      1             2.426382         0.205762  0.766181  0.515451   1.481854e-05
  initial      2             1.499963         0.183929  0.792875  0.577955   7.411130e-06
  initial      3             1.284821         0.174636  0.812881  0.644635   3.722315e-09
continued      4             1.196773         0.172788  0.812875  0.646656   3.704634e-06
continued      5             1.080910         0.174381  0.813714  0.653532   1.852782e-06
continued      6             1.051548         0.172744  0.817246  0.663576   9.305788e-10

Fisier salvat: /home/emilovici/projects/NLP-Master/Project_multi_eurlex/Results/xlm_roberta_training_history.csv
Numar epoci arhivate: 6


Nu ștergem încă folderele vechi. Mai întâi verificăm dacă modelul final poate fi încărcat independent.

Vom încărca modelul direct din xlm_roberta_best, fără să folosim obiectul Trainer sau checkpoint-urile anterioare.

De asemenea, vom verifica dacă poate genera predicții pentru un document.

In [55]:

import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BEST_MODEL_DIR = Path("../Models/xlm_roberta_best")

# Incarcam exclusiv modelul final salvat
best_tokenizer = AutoTokenizer.from_pretrained(
    str(BEST_MODEL_DIR)
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    str(BEST_MODEL_DIR)
)

best_model.eval()

# Folosim primul document din validation
sample_text = val_df.iloc[0]["text"]

inputs = best_tokenizer(
    sample_text,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = best_model(**inputs)

probabilities = torch.sigmoid(outputs.logits)

print("Model incarcat:", BEST_MODEL_DIR)
print("Numar categorii:", best_model.config.num_labels)
print("Forma predictiilor:", probabilities.shape)

assert best_model.config.num_labels == 21
assert probabilities.shape == (1, 21)
assert torch.isfinite(probabilities).all()

print("\nModelul final functioneaza independent!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model incarcat: ../Models/xlm_roberta_best
Numar categorii: 21
Forma predictiilor: torch.Size([1, 21])

Modelul final functioneaza independent!


Curățarea folderului Models

In [56]:

from pathlib import Path
import shutil

MODELS_DIR = Path("../Models").resolve()

folders_to_delete = [
    "xlm_roberta_3epochs",
    "xlm_roberta_finetuned",
    "xlm_roberta_continued"
]

best_model_dir = MODELS_DIR / "xlm_roberta_best"

# Verificam modelul pe care il pastram
assert (best_model_dir / "model.safetensors").is_file()
assert (best_model_dir / "config.json").is_file()
assert (best_model_dir / "experiment_info.json").is_file()

# Verificam istoricul antrenarii
history_path = Path(
    "../Results/xlm_roberta_training_history.csv"
).resolve()

assert history_path.is_file()

# Stergem doar directoarele specificate
for folder_name in folders_to_delete:
    folder_path = MODELS_DIR / folder_name

    if folder_path.is_dir():
        shutil.rmtree(folder_path)
        print("Sters:", folder_name)
    else:
        print("Nu exista:", folder_name)

print("\nDirectoare ramase in Models:")

for folder in sorted(MODELS_DIR.iterdir()):
    if folder.is_dir():
        print("-", folder.name)

print("\nModelul final si istoricul au fost pastrate.")


Sters: xlm_roberta_3epochs
Sters: xlm_roberta_finetuned
Sters: xlm_roberta_continued

Directoare ramase in Models:
- xlm_roberta_best

Modelul final si istoricul au fost pastrate.


Înainte să trecem la Legal-XLM-Longformer, aș vrea să terminăm corect evaluarea XLM-RoBERTa pe setul de validation.

Am calculat până acum metricile globale, dar nu știm încă care dintre cele 21 de categorii sunt clasificate bine și care sunt problematice.

Acest lucru este important pentru că am observat anterior că distribuția categoriilor este foarte dezechilibrată.

Precision, Recall și F1 pentru fiecare categorie
Vom folosi modelul final deja încărcat în best_model și obiectul val_dataset. Nu antrenăm nimic.

Celula 23 – Evaluarea individuală a categoriilor

Rulează modelul final pe întregul validation, fără antrenare. Calculează probabilitățile pentru fiecare categorie, aplică pragul de 0.5 și generează raportul individual.

In [57]:

from sklearn.metrics import classification_report, f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import torch

# Folosim modelul final deja incarcat
best_model.eval()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
best_model.to(device)

# Pregatim documentele de validation
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=data_collator
)

all_probabilities = []
all_labels = []

with torch.inference_mode():
    for batch in tqdm(val_loader, desc="Evaluare validation"):
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device)
        }

        outputs = best_model(**inputs)

        probabilities = torch.sigmoid(
            outputs.logits.float()
        ).cpu().numpy()

        all_probabilities.append(probabilities)
        all_labels.append(batch["labels"].numpy())

y_prob = np.concatenate(all_probabilities)
y_true = np.concatenate(all_labels).astype(int)
y_pred = (y_prob >= 0.5).astype(int)

# Verificam dimensiunile
assert y_true.shape == (5000, 21)
assert y_pred.shape == (5000, 21)

micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("Documente evaluate:", len(y_true))
print("Micro-F1:", round(micro_f1, 6))
print("Macro-F1:", round(macro_f1, 6))

# Verificam rezultatele fata de antrenarea anterioara
assert np.isclose(micro_f1, 0.8172456326554028, atol=1e-4)
assert np.isclose(macro_f1, 0.663576, atol=1e-4)

# Raport pentru fiecare categorie
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(21)),
    target_names=[
        str(id2label[i]) for i in range(21)
    ],
    output_dict=True,
    zero_division=0
)

print("\n=== PERFORMANTA PE CATEGORII ===")

for i in range(21):
    label = str(id2label[i])
    metrics = report[label]

    print(
        f"{label} | "
        f"Precision: {metrics['precision']:.4f} | "
        f"Recall: {metrics['recall']:.4f} | "
        f"F1: {metrics['f1-score']:.4f} | "
        f"Support: {int(metrics['support'])}"
    )


Evaluare validation:   0%|          | 0/5000 [00:00<?, ?it/s]

Documente evaluate: 5000
Micro-F1: 0.81728
Macro-F1: 0.663627

=== PERFORMANTA PE CATEGORII ===
100142 | Precision: 0.7826 | Recall: 0.2647 | F1: 0.3956 | Support: 408
100143 | Precision: 0.7865 | Recall: 0.8506 | F1: 0.8173 | Support: 944
100144 | Precision: 0.7842 | Recall: 0.8072 | F1: 0.7955 | Support: 1499
100145 | Precision: 0.7181 | Recall: 0.5219 | F1: 0.6045 | Support: 571
100146 | Precision: 0.7639 | Recall: 0.6025 | F1: 0.6737 | Support: 478
100147 | Precision: 0.9295 | Recall: 0.7985 | F1: 0.8590 | Support: 2690
100148 | Precision: 0.9010 | Recall: 0.8489 | F1: 0.8741 | Support: 1211
100149 | Precision: 0.7311 | Recall: 0.7686 | F1: 0.7494 | Support: 778
100150 | Precision: 0.7307 | Recall: 0.5437 | F1: 0.6235 | Support: 469
100151 | Precision: 1.0000 | Recall: 0.0154 | F1: 0.0303 | Support: 65
100152 | Precision: 0.8324 | Recall: 0.7034 | F1: 0.7625 | Support: 445
100153 | Precision: 0.4980 | Recall: 0.6059 | F1: 0.5467 | Support: 203
100154 | Precision: 0.9108 | Recall: 0

Avem acum prima evaluare detaliată a modelului XLM-RoBERTa pe toate cele 21 de categorii EUROVOC. Rezultatele arată că modelul clasifică foarte bine anumite domenii, dar are dificultăți serioase la altele.

Aceste rezultate sugerează că modelul identifică bine domeniile cu vocabular distinct și reprezentare consistentă în date. Aceasta este însă o ipoteză de interpretare, nu o cauză demonstrată încă.

Observații importante
Categoria „Organizații internaționale” are F1 = 0. Modelul nu a identificat corect niciun exemplu pozitiv din cele 69 existente. Din aceste metrici nu știm încă dacă nu a prezis niciodată categoria sau dacă a prezis-o doar greșit.

Categoria „Știință” are Precision = 1, dar Recall = 0,0154. Din cele 65 de exemple reale, rezultatele sunt compatibile cu o singură predicție pozitivă corectă și fără predicții fals pozitive. Așadar, Precision = 1 nu înseamnă aici că modelul clasifică bine categoria.

Categoria „Politică” are Recall = 0,2647. Modelul omite aproximativ 73,5% dintre exemplele reale din această categorie.

 Ce concluzionăm pentru proiect?
Avem un baseline cu performanță globală bună pe validation, dar cu diferențe mari între categorii.

Nu putem afirma că dezechilibrul claselor este singura cauză. Performanța poate fi influențată și de asemănarea semantică dintre categorii, calitatea etichetelor, trunchierea documentelor și pragul de clasificare.

Pentru comparația cu Legal-XLM-Longformer vom urmări în special dacă acesta îmbunătățește performanța categoriilor problematice, nu doar Micro-F1 global.

Înainte să trecem la Longformer, salvăm metricile reale, pentru a le folosi în comparația finală și în prezentare.

Celula 24 - Salvăm raportul pe categorii

In [59]:

from pathlib import Path
import pandas as pd

RESULTS_DIR = Path("../Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for i in range(21):
    label = str(id2label[i])
    metrics = report[label]

    rows.append({
        "label_id": label,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1-score"],
        "support": int(metrics["support"])
    })

category_results = pd.DataFrame(rows)

output_path = (
    RESULTS_DIR / "xlm_roberta_per_category_validation.csv"
)

category_results.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

# Verificam fisierul salvat
saved_results = pd.read_csv(
    output_path,
    dtype={"label_id": str}
)

assert len(saved_results) == 21
assert saved_results["support"].sum() == int(y_true.sum())

print(saved_results.to_string(index=False))
print("\nFisier salvat:", output_path.resolve())
print("Categorii salvate:", len(saved_results))


label_id  precision   recall       f1  support
  100142   0.782609 0.264706 0.395604      408
  100143   0.786484 0.850636 0.817303      944
  100144   0.784187 0.807205 0.795529     1499
  100145   0.718072 0.521891 0.604462      571
  100146   0.763926 0.602510 0.673684      478
  100147   0.929468 0.798513 0.859028     2690
  100148   0.900964 0.848885 0.874150     1211
  100149   0.731051 0.768638 0.749373      778
  100150   0.730659 0.543710 0.623472      469
  100151   1.000000 0.015385 0.030303       65
  100152   0.832447 0.703371 0.762485      445
  100153   0.497976 0.605911 0.546667      203
  100154   0.910828 0.837891 0.872838      512
  100155   0.686775 0.574757 0.625793      515
  100156   0.917557 0.874818 0.895678     2061
  100157   0.952593 0.958271 0.955423     2013
  100158   0.839161 0.415225 0.555556      578
  100159   0.903226 0.545455 0.680162      154
  100160   0.796767 0.607394 0.689311      568
  100161   0.923624 0.935159 0.929356     2082
  100162   0.

Celula 25 – Adăugarea denumirilor românești în raport

In [ ]:

import json
from pathlib import Path

# Folosim fisierul descarcat in notebook-ul EDA
descriptor_path = "../Data/eurovoc_descriptors.json"

with open(descriptor_path, "r", encoding="utf-8") as f:
    descriptors = json.load(f)

# Adaugam denumirile categoriilor in romana
category_results["category"] = (
    category_results["label_id"].map(
        lambda label: descriptors.get(
            str(label), {}
        ).get("ro", str(label))
    )
)

# Ne asiguram ca toate cele 21 de categorii au denumiri
missing = category_results[
    category_results["category"]
    == category_results["label_id"]
]

assert missing.empty, (
    "Lipsesc traduceri pentru: "
    + ", ".join(missing["label_id"])
)

# Reordonam coloanele
category_results = category_results[
    [
        "label_id",
        "category",
        "precision",
        "recall",
        "f1",
        "support"
    ]
]

# Actualizam fisierul CSV existent
category_results.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print(
    category_results[
        ["label_id", "category", "f1", "support"]
    ].to_string(index=False)
)

print("\nRaport actualizat:", output_path.resolve())


AttributeError: 'str' object has no attribute 'exists'